In [2]:
import pandas as pd
import json
from nltk.tokenize import RegexpTokenizer
import numpy as np
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

1. As per the JD only candidates notice period less than 60days(because 30days buy out option is there), experience in between 5 to 9, who is in India and willing to relocate candidates only considered.
2. One more information acutally needed as per the JD i.e. VISA sponsership required for the candidate or not.

However we don't have information about 2 in the dataset, hence used only 1st condition to create dataset

In [3]:
#Creating data from json file

json_file = "candidates.jsonl"

profiles = []
career_history = []
education = []
skills = []
certifications = []
languages = []
redrob_signals = []
industries = []
with open(json_file, "r", encoding="utf-8") as f:
    for line in f:
        candidate = json.loads(line)

        candidate_id = candidate["candidate_id"]

        profile = candidate.get("profile", {})
        signal = candidate.get("redrob_signals", {})
      #Filter the candidates who are not in India.
        if((profile.get("country") != "India")):
          continue
      #Filter the candidates who are not willing to relocate (There is VISA sponsership required information hence only used this).
        candLocation = profile.get("location").strip().lower()
        if (candLocation == "hyderabad" or candLocation =='pune' or candLocation =='noida' or candLocation =='mumbai' or candLocation =='delhi'):
          print("do nothing")
        elif (signal.get("willing_to_relocate") == False ):
          continue
      #Filter the candidates whoes notice period is >60 days (becuase 30days buyout option is available)
        if (signal.get("notice_period_days") > 60 ):
          continue
        exp = profile.get("years_of_experience")
        if exp < 5 or exp > 9:
          continue

        # -------------------------------
        # Candidate Profile
        # -------------------------------
        profiles.append({
            "candidate_id": candidate_id,
            "name": profile.get("anonymized_name"),
            "headline": profile.get("headline"),
            "summary": profile.get("summary"),
            "location": profile.get("location"),
            "country": profile.get("country"),
            "years_of_exp": profile.get("years_of_experience"),
            "current_title": profile.get("current_title"),
            "current_company": profile.get("current_company"),
            "current_company_size": profile.get("current_company_size"),
            "current_industry": profile.get("current_industry")
        })

        # Industry from current profile
        industries.append({
            "candidate_id": candidate_id,
            "Type_of_org": profile.get("current_industry"),
            "company_size": profile.get("current_company_size"),

        })
        candidate_id = candidate.get("candidate_id")
        companies = []
        titles = []
        start_dates = []
        end_dates = []
        industries = []
        company_sizes = []
        experience_descriptions = []

        # -------------------------------
        # Career History
        # -------------------------------
        for exp in candidate.get("career_history", []):
          # Convert months to years
          duration_months = exp.get("duration_months", 0)


          try:
               years = round(float(duration_months) / 12, 1)
          except:
               years = 0

          description = exp.get("description", "")

          enhanced_description = (
             f"{description} Worked here for about {years} years."
          ).strip()

          companies.append(str(exp.get("company", "")))
          titles.append(str(exp.get("title", "")))
          start_dates.append(str(exp.get("start_date", "")))
          end_dates.append(str(exp.get("end_date", "")))
          industries.append(str(exp.get("industry", "")))
          company_sizes.append(str(exp.get("company_size", "")))
          experience_descriptions.append(enhanced_description)

        career_history.append({
            "candidate_id": candidate_id,
            "companies": ", ".join(companies),
            "titles": ", ".join(titles),
            "start_dates": ", ".join(start_dates),
            "end_dates": ", ".join(end_dates),
            "industries": ", ".join(industries),
            "company_sizes": ", ".join(company_sizes),
             "career_history": " | ".join(experience_descriptions)
            })


            # Industry from career history
        industries.append({
                "candidate_id": candidate_id,
                "Type_of_org": exp.get("industry"),
                "company_size": exp.get("company_size"),
            })

        # -------------------------------
        # Education
        # -------------------------------
        for edu in candidate.get("education", []):
            education.append({
                "candidate_id": candidate_id,
                "institution": edu.get("institution"),
                "degree": edu.get("degree"),
                "field_of_study": edu.get("field_of_study"),
                "start_year": edu.get("start_year"),
                "end_year": edu.get("end_year"),
                "grade": edu.get("grade"),
                "tier": edu.get("tier")
            })

        # -------------------------------
        # Skills
        # -------------------------------
        candidate_skills = []
        for skill in candidate.get("skills", []):
          try:
              years = round(float(skill.get("duration_months", 0)) / 12, 1)
          except:
              years = 0

          skill_text = (
            f"{skill.get('name', '')} "
            f"({skill.get('proficiency', 'Unknown')}, "
            f"{skill.get('endorsements', 0)} endorsements, "
            f"{years} years)"
          )

          candidate_skills.append(skill_text)


        skills.append({
          "candidate_id": candidate_id,
          "skills": ", ".join(candidate_skills)
       })

        # -------------------------------
        # Certifications
        # -------------------------------
        for cert in candidate.get("certifications", []):
            certifications.append({
                "candidate_id": candidate_id,
                "name": cert.get("name"),
                "issuer": cert.get("issuer"),
                "year": cert.get("year")
            })

        # -------------------------------
        # Languages
        # -------------------------------
        for lang in candidate.get("languages", []):
            languages.append({
                "candidate_id": candidate_id,
                "language": lang.get("language"),
                "proficiency": lang.get("proficiency")
            })

        # -------------------------------
        # Redrob Signals
        # -------------------------------


        redrob_signals.append({
            "candidate_id": candidate_id,
            "profile_completeness_score": signal.get("profile_completeness_score"),
            "signup_date": signal.get("signup_date"),
            "last_active_date": signal.get("last_active_date"),
            "open_to_work_flag": signal.get("open_to_work_flag"),
            "profile_views_received_30d": signal.get("profile_views_received_30d"),
            "applications_submitted_30d": signal.get("applications_submitted_30d"),
            "recruiter_response_rate": signal.get("recruiter_response_rate"),
            "avg_response_time_hours": signal.get("avg_response_time_hours"),
            "skill_assessment_scores": json.dumps(
                signal.get("skill_assessment_scores", {})
            ),
            "connection_count": signal.get("connection_count"),
            "endorsements_received": signal.get("endorsements_received"),
            "notice_period_days": signal.get("notice_period_days"),
            "expected_min_salary_lpa":
                signal.get("expected_salary_range_inr_lpa", {}).get("min"),
            "expected_max_salary_lpa":
                signal.get("expected_salary_range_inr_lpa", {}).get("max"),
            "preferred_work_mode": signal.get("preferred_work_mode"),
            "willing_to_relocate": signal.get("willing_to_relocate"),
            "github_activity_score": signal.get("github_activity_score"),
            "search_appearance_30d": signal.get("search_appearance_30d"),
            "saved_by_recruiters_30d": signal.get("saved_by_recruiters_30d"),
            "interview_completion_rate": signal.get("interview_completion_rate"),
            "offer_acceptance_rate": signal.get("offer_acceptance_rate"),
            "verified_email": signal.get("verified_email"),
            "verified_phone": signal.get("verified_phone"),
            "linkedin_connected": signal.get("linkedin_connected")
        })






# ==========================================
# Create CSV files
# ==========================================
datasetPath = ""
pd.DataFrame(profiles).to_csv(
    datasetPath+"candidate_profile.csv", index=False)

pd.DataFrame(career_history).to_csv(
    datasetPath+"candidate_career_history.csv", index=False)

pd.DataFrame(education).to_csv(
    datasetPath+"candidate_education.csv", index=False)

pd.DataFrame(skills).to_csv(
    datasetPath+"candidate_skills.csv", index=False)

pd.DataFrame(certifications).to_csv(
    datasetPath+"candidate_certifications.csv", index=False)

pd.DataFrame(languages).to_csv(
    datasetPath+"candidate_languages.csv", index=False)

pd.DataFrame(redrob_signals).to_csv(
    datasetPath+"candidate_redrob_signals.csv", index=False)

pd.DataFrame(industries).to_csv(
    datasetPath+"candidate_org.csv", index=False)
print("All CSV files generated successfully.")

All CSV files generated successfully.


Creating Vector database for the candidates based on the skills, summary and career history

In [4]:
CANDIDATE_PROFILE = 'candidate_profile.csv'
CANDIDATE_SKILLS = 'candidate_skills.csv'
CANDIDATE_CAREER_HISTORY = 'candidate_career_history.csv'
CANDIDATE_REDROB_SIGNALS= 'candidate_redrob_signals.csv'

In [8]:
FAISS_INDEX_FILE = "candSummaryvector_db.faiss"

METADATA_FILE = "candSummarymetadata.pkl"

MODEL_NAME = "all-MiniLM-L6-v2"

In [9]:



# =====================================================
# LOAD FILES
# =====================================================

profile_df = pd.read_csv(CANDIDATE_PROFILE)

skills_df = pd.read_csv(CANDIDATE_SKILLS)

career_df = pd.read_csv(CANDIDATE_CAREER_HISTORY)



print("Profile Rows :", len(profile_df))
print("Skills Rows  :", len(skills_df))
print("Career Rows  :", len(career_df))

# =====================================================
# KEEP REQUIRED COLUMNS
# =====================================================

skills_df = skills_df[[
    "candidate_id",
    "skills"
]]

career_df = career_df[[
    "candidate_id",
    "companies",
    "titles",
    "industries",
    "career_history"
]]

# =====================================================
# MERGE DATA
# =====================================================

df = profile_df.merge(
    skills_df,
    on="candidate_id",
    how="left"
)

df = df.merge(
    career_df,
    on="candidate_id",
    how="left"
)

# =====================================================
# HANDLE NULLS
# =====================================================

columns_to_fill = [
    'headline',
    "summary",
    "skills",
    "companies",
    "titles",
    "industries",
    "career_history"
]

for col in columns_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna("")

# =====================================================
# CREATE COMBINED TEXT
# =====================================================

df["combined_text"] = (
    'Current designation: ' + df['headline'].astype(str)+
    "Summary: " + df["summary"].astype(str) +
    "\nSkills: " + df["skills"].astype(str) +
    "\nCompanies: " + df["companies"].astype(str) +
    "\nJob Titles: " + df["titles"].astype(str) +
    "\nIndustries: " + df["industries"].astype(str) +
    "\nCareer History: " + df["career_history"].astype(str)
)

texts = df["combined_text"].tolist()

print("Documents:", len(texts))

# =====================================================
# LOAD MODEL
# =====================================================

print("Loading embedding model...")

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE EMBEDDINGS
# =====================================================

print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=256,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)

print("Embedding Shape:", embeddings.shape)

# =====================================================
# CREATE FAISS INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

# =====================================================
# SAVE INDEX
# =====================================================

faiss.write_index(
    index,
    FAISS_INDEX_FILE
)

print("FAISS index saved")

# =====================================================
# SAVE METADATA
# =====================================================

metadata = df.to_dict("records")

with open(METADATA_FILE, "wb") as f:
    pickle.dump(metadata, f)

print("Metadata saved")
print("Done!")

Profile Rows : 2751
Skills Rows  : 2751
Career Rows  : 2751
Documents: 2751
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding Shape: (2751, 384)
Vectors Stored: 2751
FAISS index saved
Metadata saved
Done!


Getting Sematic score of candidates skills, summary and previous work experience with JD

In [11]:
JSON_FILE = "job_requirements.json"

In [14]:
# =====================================================
# LOAD JOB REQUIREMENTS JSON
# =====================================================

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

# =====================================================
# EXTRACT REQUIRED INFORMATION
# =====================================================

# Core Skills
core_skills = [
    skill["skill_id"].replace("_", " ")
    for skill in jd["preferred_skills"]["core_required"]
]

# Secondary Skills
secondary_skills = [
    skill["skill_id"].replace("_", " ")
    for skill in jd["preferred_skills"]["secondary_skills"]
]

# Experience Areas
primary_exp = jd["primary_experience_areas"]
secondary_exp = jd["secondary_experience_areas"]
not_preferred_exp = jd["not_preferred_experience"]

# Experience
experience = jd["experience_duration"]
ideal_experience = jd["ideal_experience_duration"]

# Job Title
job_title = jd["current_job_title"]

# Industry
industry = jd["industry"]


# =====================================================
# BUILD SEMANTIC QUERY
# =====================================================

jd_text = f"""
Hiring for {job_title}

Industry:
{industry}

Experience Required:
{experience} years

Ideal Experience:
{ideal_experience} years

Primary Experience Areas:
{primary_exp}

Secondary Experience Areas:
{secondary_exp}

Mandatory Skills:
{", ".join(core_skills)}

Preferred Skills:
{", ".join(secondary_skills)}

Not Preferred Experience:
{not_preferred_exp}
"""

print("=" * 100)
print("JD QUERY")
print("=" * 100)
print(jd_text)

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE JD EMBEDDING
# =====================================================

jd_embedding = model.encode(
    [jd_text],
    normalize_embeddings=True
).astype(np.float32)

JD QUERY

Hiring for Senior AI Engineer

Industry:
AI

Experience Required:
5,9 years

Ideal Experience:
6,8 years

Primary Experience Areas:
production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5, or similar), production level operational experience with vector databases or hybrid search infrastructure (Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS, or similar), Strong Python, Hands-on experience designing evaluation frameworks for ranking systems (NDCG, MRR, MAP, offline-to-online correlation, A/B test interpretation), LLM fine-tuning experience (LoRA, QLoRA, PEFT), Experience with learning-to-rank models (XGBoost-based or neural)

Secondary Experience Areas:
Prior exposure to HR-tech, recruiting tech, or marketplace products, Background in distributed systems or large-scale inference optimization, Open-source contributions in the AI/ML space

Mandatory Skills:
EMBEDDINGS, RETRIEVAL SYSTEMS, VECTOR DB, H

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
# =====================================================
# LOAD FAISS
# =====================================================

index = faiss.read_index(FAISS_INDEX_FILE)

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("Candidates:", len(metadata))

# =====================================================
# SEARCH ALL CANDIDATES
# =====================================================

k = index.ntotal

scores, indices = index.search(
    jd_embedding,
    k
)

Candidates: 2751


Generating sematic score from vector DB and JD

In [16]:
# =====================================================
# BUILD RESULT CSV
# =====================================================

results = []

for rank, idx in enumerate(indices[0]):

    similarity = float(scores[0][rank])

    # cosine similarity -> score 0-1
    semantic_score = (similarity + 1) / 2


    candidate = metadata[idx]

    results.append({
        "candidate_id": candidate.get("candidate_id"),
        "semantic_similarity": round(similarity, 4),
        "semantic_score": semantic_score
    })

# =====================================================
# SAVE OUTPUT
# =====================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "semantic_score",
    ascending=False
)

OUTPUT_FILE = "candidate_semantic_scores.csv"

result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(result_df.head(20))

Saved: /content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores.csv
    candidate_id  semantic_similarity  semantic_score
0   CAND_0076251               0.7600        0.880001
1   CAND_0041610               0.7481        0.874072
2   CAND_0098952               0.7343        0.867138
3   CAND_0067866               0.7299        0.864951
4   CAND_0067643               0.7294        0.864719
5   CAND_0043860               0.7289        0.864466
6   CAND_0018549               0.7288        0.864401
7   CAND_0068811               0.7268        0.863418
8   CAND_0020877               0.7266        0.863304
9   CAND_0032216               0.7265        0.863259
10  CAND_0005260               0.7244        0.862198
11  CAND_0048558               0.7227        0.861364
12  CAND_0025923               0.7224        0.861183
13  CAND_0042029               0.7203        0.860162
14  CAND_0092568               0.7200        0.860003
15  CAND_0046525            

Creating a score for redrob signals. For candidate Notice period, is candidate active or inactive, candidate acceptance rate, candidate prefered work mode(flexible, Hybrid,onsite or Remote), candidate is open to work or not


In [17]:
df = pd.read_csv(CANDIDATE_REDROB_SIGNALS)

# =====================================================
# LAST ACTIVE DATE SCORE
# More recent activity = higher score
# =====================================================

df['last_active_date'] = pd.to_datetime(
    df['last_active_date'],
    errors='coerce'
)

today = pd.Timestamp.today().normalize()

days_since_active = (
    today - df['last_active_date']
).dt.days

# Cap at 365 days
days_since_active = days_since_active.clip(
    lower=0,
    upper=365
)

HALF_LIFE = 180  # days

df['last_active_score'] = (
    100 * np.exp(-days_since_active / HALF_LIFE)
).clip(0, 100).round(2)

# Missing dates
df['last_active_score'] = df['last_active_score'].fillna(0)

# =====================================================
# OPEN TO WORK SCORE
# =====================================================

df['open_to_work_score'] = np.where(
    df['open_to_work_flag']
      .astype(str)
      .str.lower()
      .isin(['true', 'yes', '1', 'y']),
    100,
    0
)



# =====================================================
# PREFERRED WORK MODE SCORE
#
# remote= -100 hybrid = flexible = onsite = 100% because work mode is Hybrid.
# =====================================================

work_mode_scores = {
    'remote': 0,
    'hybrid': 100,
    'flexible': 100,
    'onsite': 100
}

df['preferred_work_mode_score'] = (
    df['preferred_work_mode']
      .astype(str)
      .str.strip()
      .str.lower()
      .map(work_mode_scores)
      .fillna(0)
)

# =====================================================
# OFFER ACCEPTANCE RATE SCORE
# =====================================================

df['offer_acceptance_rate'] = pd.to_numeric(
    df['offer_acceptance_rate'],
    errors='coerce'
)

df['offer_acceptance_score'] = (
    (df['offer_acceptance_rate']*100)
).round(2)

df['final_candidate_signal_score'] = (
      0.50 * df['last_active_score']
    + 0.15 * df['preferred_work_mode_score']
    + 0.15 * df['open_to_work_score']
    + 0.20 * df['offer_acceptance_score']
)

# =====================================================
# OUTPUT FILE
# =====================================================

output_df = df[[
    'candidate_id',
    'last_active_score',
    'open_to_work_score',
    'preferred_work_mode_score',
    'offer_acceptance_score',
    'final_candidate_signal_score'
]]


OUTPUT_FILE = "candidate_redrob_signals_scored.csv"

output_df.to_csv(
    OUTPUT_FILE,
    index=False
)

# =====================================================
# SUMMARY
# =====================================================

print("Output Shape:", output_df.shape)

print("\nSample Results:")
print(output_df.head())

print(f"\nScored file saved as: {OUTPUT_FILE}")

Output Shape: (2751, 6)

Sample Results:
   candidate_id  last_active_score  open_to_work_score  \
0  CAND_0000007              82.33                   0   
1  CAND_0000024              41.11                   0   
2  CAND_0000031              81.87                 100   
3  CAND_0000061              34.99                   0   
4  CAND_0000088              49.94                   0   

   preferred_work_mode_score  offer_acceptance_score  \
0                        100                  -100.0   
1                        100                  -100.0   
2                        100                    38.0   
3                        100                  -100.0   
4                        100                    69.0   

   final_candidate_signal_score  
0                        36.165  
1                        15.555  
2                        78.535  
3                        12.495  
4                        53.770  

Scored file saved as: /content/drive/MyDrive/Hackthon/IndiaRuns/vers

Creating final score from semantic score and redrob signal score

In [18]:
signal_df = pd.read_csv("candidate_redrob_signals_scored.csv")
semantic_df = pd.read_csv("candidate_semantic_scores.csv")

# Keep required columns from signal file
signal_df = signal_df[
    ['candidate_id', 'final_candidate_signal_score']
]

# Merge on candidate_id
final_df = semantic_df.merge(
    signal_df,
    on='candidate_id',
    how='left'
)

# Normalize signal score (-100 to 100 -> -1 to 1)
final_df['signal_score_normalized'] = (
    final_df['final_candidate_signal_score'] / 100
)

# Final Score cosidered only 20% from redrob signal score becuase two things are important from that file rest should be considered from sematic score
final_df['final_score'] = (
      0.80 * final_df['semantic_score']
    + 0.20 * final_df['signal_score_normalized']
).round(4)

# Sort by final score descending
final_df = final_df.sort_values(
    by='final_score',
    ascending=False
)

# Keep required columns
final_output = final_df[
    [
        'candidate_id',
        'semantic_score',
        'final_candidate_signal_score',
        'final_score'
    ]
]

# Save
final_output.to_csv(
    "candidate_final_scores.csv",
    index=False
)

print(final_output.head(10))

    candidate_id  semantic_score  final_candidate_signal_score  final_score
5   CAND_0043860        0.864466                        87.395       0.8664
7   CAND_0068811        0.863418                        84.860       0.8605
6   CAND_0018549        0.864401                        83.230       0.8580
0   CAND_0076251        0.880001                        74.560       0.8531
11  CAND_0048558        0.861364                        80.825       0.8507
15  CAND_0046525        0.859863                        80.310       0.8485
9   CAND_0032216        0.863259                        77.650       0.8459
19  CAND_0081852        0.857184                        78.910       0.8436
63  CAND_0005477        0.844002                        83.735       0.8427
22  CAND_0017960        0.855496                        78.035       0.8405


Creating final file

In [19]:
df = pd.read_csv("candidate_final_scores.csv")

# Take first 100 rows
top_100 = df.head(100).copy()

# Create rank column
top_100["rank"] = range(1, len(top_100) + 1)

# Select required columns and rename final_score to score
result_df = top_100[["candidate_id", "rank", "final_score"]].rename(
    columns={"final_score": "score"}
)


# Save to a new CSV file
result_df.to_csv("top_100_candidates.csv", index=False)

print("Top 100 candidates with there score against job description is saved...")

Top 100 candidates with there score against job description is saved...


Creating the reason for top 100 candidates.

filter the profile file, skill, career history and redrob signal files.



In [20]:
top_candidates = pd.read_csv("top_100_candidates.csv")

# Profile file to filter
candidateProfile_df = pd.read_csv("candidate_profile.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateProfile_df[
    candidateProfile_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("filtered_candidateprofile_file.csv", index=False)

# Career history file to filter
candidateCareerHistory_df = pd.read_csv("candidate_career_history.csv")

filtered_df = candidateCareerHistory_df[
    candidateCareerHistory_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("filtered_candidatecareerhistory_file.csv", index=False)

#redrob signal file to filter
candidateredrobsignal_df = pd.read_csv("candidate_redrob_signals.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateredrobsignal_df[
    candidateredrobsignal_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("filtered_redrobsignal_file.csv", index=False)

#skills file to filter
candidateskills_df = pd.read_csv("candidate_skills.csv")

# Assuming both files have a column named 'candidate_id'
filtered_df = candidateskills_df[
    candidateskills_df['candidate_id'].isin(top_candidates['candidate_id'])
]

# Save filtered file
filtered_df.to_csv("filtered_skills_file.csv", index=False)


In [27]:
# =====================================================
# CONFIG
# =====================================================

TOP100_FILE = "top_100_candidates.csv"
PROFILE_FILE = "filtered_candidateprofile_file.csv"
CAREER_FILE = "filtered_candidatecareerhistory_file.csv"
SIGNAL_FILE = "filtered_redrobsignal_file.csv"
SKILLS_FILE = "filtered_skills_file.csv"
OUTPUT_FILE= "submission.csv"

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [28]:


print("Loading files...")

top100 = pd.read_csv(TOP100_FILE)
profile = pd.read_csv(PROFILE_FILE)
career = pd.read_csv(CAREER_FILE)
signals = pd.read_csv(SIGNAL_FILE)
skills = pd.read_csv(SKILLS_FILE)

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

print("Files loaded.")

# ==========================================================
# SMALL JD SUMMARY
# ==========================================================

jd_summary = {
    "job_title": jd["current_job_title"],

    "experience": jd["experience_duration"],
    "ideal_experience": jd["ideal_experience_duration"],

    "industry": jd["industry"],

    "primary_experience_areas": jd["primary_experience_areas"],
    "secondary_experience_areas": jd["secondary_experience_areas"],

    "required_skills": [
        skill["skill_id"]
        for skill in jd["preferred_skills"]["core_required"]
    ],

    "preferred_skills": [
        skill["skill_id"]
        for skill in jd["preferred_skills"]["secondary_skills"]
    ],

    "not_preferred_experience": jd["not_preferred_experience"]
}

# ==========================================================
# LOAD MODEL
# ==========================================================


print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded.")

# ==========================================================
# GENERATE REASON
# ==========================================================

def generate_reason(candidate_id):

    profile_df = profile[
        profile["candidate_id"] == candidate_id
    ]

    career_df = career[
        career["candidate_id"] == candidate_id
    ]

    signal_df = signals[
        signals["candidate_id"] == candidate_id
    ]

    skills_df = skills[
        skills["candidate_id"] == candidate_id
    ]

    # ----------------------------
    # Candidate Summary
    # ----------------------------

    current_title = ""
    years_exp = ""

    if not profile_df.empty:

        if "current_title" in profile_df.columns:
            current_title = profile_df.iloc[0]["current_title"]

        if "years_of_exp" in profile_df.columns:
            years_exp = profile_df.iloc[0]["years_of_exp"]

    industries = []

    if (
        not career_df.empty
        and "industries" in career_df.columns
    ):
        industries = career_df["industries"].dropna().unique().tolist()

    offer_rate = ""

    if (
        not signal_df.empty
        and "offer_acceptance_rate" in signal_df.columns
    ):
        offer_rate = signal_df.iloc[0]["offer_acceptance_rate"]

    candidate_skills = []

    # Change "skill_name" if your column name is different
    if (
        not skills_df.empty
        and "skills" in skills_df.columns
    ):
        candidate_skills = (
            skills_df["skills"]
            .dropna()
            .unique()
            .tolist()
        )

    candidate_summary = {
        "current_title": current_title,
        "years_of_experience": years_exp,
        "industries": industries,
        "offer_acceptance_rate": offer_rate,
        "skills": candidate_skills
    }

    # ----------------------------
    # Prompt
    # ----------------------------
    prompt = f"""You are an experienced recruiter. Write one professional hiring reason in 100-120 words. Use ONLY the candidate information below.
    Job Requirements:
    {jd_summary}
    Candidate Information: {candidate_summary}
    Mention:
    - current title
    - years of experience
    - matching skills
    - relevant industry/domain experience
    Example:
    This candidate is currently working as an AI Engineer with 6 years of experience. They possess strong skills in LLM and RAG and have experience in product-based companies, making them a strong fit for this role.
    Return only the final sentence."""

    messages = [
        {
            "role": "system",
            "content": "You are an expert recruiter."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
      terminators = [
      tokenizer.eos_token_id,
      tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]
    outputs = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.3,
      top_p=0.9,
      repetition_penalty=1.1,
      pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs.input_ids.shape[1]:]
    reason = tokenizer.decode(
    generated,
    skip_special_tokens=True

    )

    return reason


# ==========================================================
# GENERATE FOR TOP 100
# ==========================================================

reasons = []

total = len(top100)

for i, row in top100.iterrows():

    candidate_id = row["candidate_id"]

    print(f"{i+1}/{total} : {candidate_id}")

    try:
        reason = generate_reason(candidate_id)
        print("reason",reason)
    except Exception as e:
        print(e)
        reason = ""

    reasons.append(reason)

# ==========================================================
# SAVE
# ==========================================================

top100["reasoning"] = reasons

top100.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nCompleted Successfully.")
print("Saved:", OUTPUT_FILE)

Loading files...
Files loaded.
Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded.
1/100 : CAND_0043860
reason This candidate is currently working as a Junior ML Engineer with 6.1 years of experience. They possess strong skills in embedding-based retrieval systems, vector databases, and hands-on experience with production-level operations, making them a strong fit for the Senior AI Engineer position.
2/100 : CAND_0068811
reason This candidate is currently working as an Applied ML Engineer with 8 years of experience, possessing strong skills in data science, vector search, speech recognition, and other relevant areas. Their expertise aligns perfectly with the Senior AI Engineer position in the AI industry, making them a highly qualified candidate for this role.
3/100 : CAND_0018549
reason This candidate is currently working as a Recommendation Systems Engineer with over 6.8 years of experience. Their expertise in Computer Vision, Weaviate, Kubernetes, Elasticsearch, and other related technologies align perfectly with the Senior AI Engineer position in th